In [7]:
#Task 4.1

import sqlite3

conn = sqlite3.connect("USER_DATA1.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE User;") #remove the table so that program can be tested multiple times
cursor.execute("DROP TABLE Credit")

cursor.execute("""CREATE TABLE IF NOT EXISTS `User` (
	`user_id`	TEXT,
	`name`	TEXT,
	`email`	TEXT,
	`hash`	TEXT,
	PRIMARY KEY(`user_id`)
);""")

cursor.execute("""CREATE TABLE IF NOT EXISTS `Credit` (
	`user_id`	TEXT,
	`dt_transaction`	TEXT,
	`type`	TEXT,
	`amt`	REAL,
	PRIMARY KEY(`user_id`,`dt_transaction`),
	FOREIGN KEY(`user_id`) REFERENCES `User`(`user_id`)
);""")

file = open("users.txt",'r')
file.readline() #discard first line: user_id,name,email,hash
count = 0
for line in file:
    user_id,name,email,hash = line.strip().split(',')
    cursor.execute("""
                    INSERT INTO User (user_id,name,email,hash)
                    VALUES (?,?,?,?)
    """, (user_id,name,email,hash))
    count += 1
print(f"Inserted {count} users.")
file.close()

file = open("credits.txt",'r')
file.readline() #discard first line: user_id,name,email,hash
count = 0
for line in file:
    user_id,dt_transaction,type,amt = line.strip().split(',')
    cursor.execute("""
                    INSERT INTO Credit (user_id,dt_transaction,type,amt)
                    VALUES (?,?,?,?)
    """, (user_id,dt_transaction,type,amt))
    count += 1
print(f"Inserted {count} transactions.")
file.close()


conn.commit()


Inserted 15 users.
Inserted 122 transactions.


In [8]:
#Task 4.2
def validate_card(card_number):
    digits = [int(digit) for digit in card_number]
    
    for i in range(len(digits) - 2, -1, -2):
        digits[i] *= 2
        if digits[i] > 9:  # 2-digit number after doubling
            digits[i] = digits[i] // 10 + digits[i] % 10
    
    total = sum(digits)
    return total % 10 == 0  # check divisibility by 10

# Test
cases = [
    4532840913491054,
    5555734352999785,  
    378282904122908,
    4111025640404402,
    5105207030955616
]

for card in cases:
    if validate_card(str(card)):
        print(f"{card} is valid.")
    else:
        print(f"{card} is invalid.")

4532840913491054 is valid.
5555734352999785 is valid.
378282904122908 is valid.
4111025640404402 is invalid.
5105207030955616 is invalid.


In [13]:
#Task 4.3
def check_user(user_id):
    cursor.execute("""
                   SELECT COUNT(*) FROM User
                   WHERE user_id = ?
                   """, (user_id,))
    count = cursor.fetchone()[0]
    return count > 0

# Test
users = ["among-us", "among-them", "i-like-fortnite"]

for user in users:
    print(f"{user} exists: {check_user(user)}")

among-us exists: False
among-them exists: False
i-like-fortnite exists: False


In [15]:
#Task 4.4
import hashlib

def update_card(user_id, new_num):
    # validation
    if not check_user(user_id):
        return f"User {user_id} does not exist."
    
    if not validate_card(new_num):
        return f"Card number {new_num} is invalid."
    
    # create salted hash
    salt = "@SR_SUP3R_S3CURE_S4LT!"
    salted_card = salt + new_num
    card_hash = hashlib.sha3_256(salted_card.encode()).hexdigest()
    
    # update database
    cursor.execute("""
                   UPDATE User
                   SET hash = ?
                   WHERE user_id = ?
                   """, (card_hash, user_id))
    conn.commit()
    
    return f"Card number hash updated for {user_id}."


def delete_data(user_id):
    if not check_user(user_id):
        return f"User {user_id} does not exist."
    
    # delete from Credit table first
    cursor.execute("""
                   DELETE FROM Credit
                   WHERE user_id = ?
                   """, (user_id,))
    
    # then delete from User table
    cursor.execute("""
                   DELETE FROM User
                   WHERE user_id = ?
                   """, (user_id,))
    conn.commit()
    
    return f"Data for user {user_id} deleted."


# Test
# Update operations
print(update_card("swimmerz", "30569309025904"))
print(update_card("koh-world", "1122334455667788"))
print()

# # Delete operations
# print(delete_data("among-us"))
# print(delete_data("i-like-fortnite"))
# print(delete_data("brawlstars"))

Card number hash updated for swimmerz.
Card number 1122334455667788 is invalid.

